In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, ExtraTreesClassifier
import warnings

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
N_SPLITS = 10 

train_df = pd.read_csv("train_assignment.csv")
test_df = pd.read_csv("test_assignment.csv")

def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["cu_total_enrolled"] = df["cu1_enrolled"] + df["cu2_enrolled"]
    df["cu_total_approved"] = df["cu1_approved"] + df["cu2_approved"]
    df["cu_total_evaluations"] = df["cu1_evaluations"] + df["cu2_evaluations"]
    eps = 1e-5
    df["approval_to_eval_ratio"] = df["cu_total_approved"] / (df["cu_total_evaluations"] + eps)
    df["approval_to_enrolled_ratio"] = df["cu_total_approved"] / (df["cu_total_enrolled"] + eps)
    df["cu_grade_mean"] = (df["cu1_grade"] + df["cu2_grade"]) / 2.0
    df["cu_grade_diff"] = df["cu2_grade"] - df["cu1_grade"]
    df["cu_approved_diff"] = df["cu2_approved"] - df["cu1_approved"]
    df["financial_distress"] = (df["debtor_flag"] == 1) & (df["tuition_up_to_date_flag"] == 0)
    df["financial_distress"] = df["financial_distress"].astype(int)
    df["risk_factor"] = df["age_at_enrollment"] * (df["debtor_flag"] + 1)

    return df

train_fe = add_features(train_df)
test_fe = add_features(test_df)

target_col = "Target"
id_col = "id"

cat_cols = [c for c in train_fe.columns if c.endswith('_code') or c.endswith('_flag')]

for c in cat_cols:
    train_fe[c] = train_fe[c].astype(int).astype('category')
    test_fe[c] = test_fe[c].astype(int).astype('category')

feature_cols = [c for c in train_fe.columns if c not in [target_col, id_col]]

X = train_fe[feature_cols]
X_test = test_fe[feature_cols]

le = LabelEncoder()
y = le.fit_transform(train_fe[target_col])
n_classes = len(le.classes_)

cat_indices = [X.columns.get_loc(c) for c in cat_cols]

def get_models():
    return {
        "xgb": XGBClassifier(
            objective="multi:softprob", n_estimators=1500, learning_rate=0.03,
            max_depth=6, subsample=0.8, colsample_bytree=0.7,
            random_state=RANDOM_STATE, enable_categorical=True, n_jobs=-1,
            tree_method="hist"
        ),
        "lgbm": LGBMClassifier(
            n_estimators=2000, learning_rate=0.03, max_depth=8, num_leaves=63,
            subsample=0.8, colsample_bytree=0.7, random_state=RANDOM_STATE, 
            n_jobs=-1, verbose=-1, categorical_feature=cat_indices
        ),
        "cat": CatBoostClassifier(
            iterations=2000, learning_rate=0.04, depth=6,
            cat_features=cat_cols, random_seed=RANDOM_STATE, verbose=0, task_type="CPU"
        ),
        "hgb": HistGradientBoostingClassifier(
            learning_rate=0.03, max_iter=1500, max_depth=7,
            categorical_features=cat_indices, random_state=RANDOM_STATE
        ),
        "et": ExtraTreesClassifier( # Added back for variance reduction
            n_estimators=1000, max_depth=15, max_features="sqrt", 
            min_samples_split=5, random_state=RANDOM_STATE, n_jobs=-1
        )
    }
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
model_names = list(get_models().keys())

oof_probas = {name: np.zeros((len(X), n_classes)) for name in model_names}
test_fold_probas = {name: [] for name in model_names}

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), start=1):
    print(f"Training Fold {fold}/{N_SPLITS}...")
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]
    
    # Handle ExtraTrees (does not support category dtype natively)
    X_tr_et = X_tr.copy()
    X_va_et = X_va.copy()
    X_test_et = X_test.copy()
    for c in cat_cols:
        X_tr_et[c] = X_tr_et[c].astype(int)
        X_va_et[c] = X_va_et[c].astype(int)
        X_test_et[c] = X_test_et[c].astype(int)

    fold_models = get_models()
    
    for name, model in fold_models.items():
        if name == "et":
            model.fit(X_tr_et, y_tr)
            oof_probas[name][va_idx] = model.predict_proba(X_va_et)
            test_fold_probas[name].append(model.predict_proba(X_test_et))
        else:
            model.fit(X_tr, y_tr)
            oof_probas[name][va_idx] = model.predict_proba(X_va)
            test_fold_probas[name].append(model.predict_proba(X_test))

test_probas_mean = {name: np.mean(test_fold_probas[name], axis=0) for name in model_names}

# ── Monte Carlo Accuracy Optimizer ───────────────────────────────────────────
print("\n Running Monte Carlo Optimization to Maximize Target Accuracy...")
oof_list = [oof_probas[name] for name in model_names]

best_acc = 0
best_weights = None
np.random.seed(42)
random_weights = np.random.dirichlet(np.ones(len(model_names)), size=15000)

for weights in random_weights:
    blend_prob = sum(weights[i] * oof_list[i] for i in range(len(model_names)))
    preds = np.argmax(blend_prob, axis=1)
    acc = accuracy_score(y, preds)
    
    if acc > best_acc:
        best_acc = acc
        best_weights = weights

print(f"\n Optimal Weights Found:")
for name, w in zip(model_names, best_weights):
    print(f"  - {name}: {w:.4f}")
print(f" Optimized OOF Accuracy: {best_acc:.5f}")

final_test_prob = sum(best_weights[i] * test_probas_mean[model_names[i]] for i in range(len(model_names)))
test_pred = le.inverse_transform(np.argmax(final_test_prob, axis=1))

submission = pd.DataFrame({"id": test_df["id"], "Target": test_pred})
submission.to_csv("submission_max_acc.csv", index=False)
print(f"\nSaved submission_max_acc.csv")